In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
%cd /content/drive/MyDrive/inventory-optimization-ml-or

/content/drive/MyDrive/inventory-optimization-ml-or


In [ ]:
!ls data/processed

sample_100_skus.csv  sample_100_skus_long.csv


In [ ]:
import pandas as pd
import numpy as np

df = pd.read_csv("data/processed/sample_100_skus_long.csv")
calendar = pd.read_csv("data/raw/calendar.csv")

print("Long data shape:", df.shape)
print("Calendar shape:", calendar.shape)

df.head()

Long data shape: (200865, 12)
Calendar shape: (1969, 14)


,id,item_id,dept_id,cat_id,store_id,state_id,avg_demand,std_demand,cv,zero_demand_ratio,sales,day
0,FOODS_1_022_CA_4_validation,FOODS_1_022,FOODS_1,FOODS,CA_4,CA,0.095661,0.398411,4.164373,0.928385,0,1
1,FOODS_1_022_CA_4_validation,FOODS_1_022,FOODS_1,FOODS,CA_4,CA,0.095661,0.398411,4.164373,0.928385,0,2
2,FOODS_1_022_CA_4_validation,FOODS_1_022,FOODS_1,FOODS,CA_4,CA,0.095661,0.398411,4.164373,0.928385,0,3
3,FOODS_1_022_CA_4_validation,FOODS_1_022,FOODS_1,FOODS,CA_4,CA,0.095661,0.398411,4.164373,0.928385,0,4
4,FOODS_1_022_CA_4_validation,FOODS_1_022,FOODS_1,FOODS,CA_4,CA,0.095661,0.398411,4.164373,0.928385,0,5


In [ ]:
calendar_small = calendar[
    [
        'd',
        'date',
        'wm_yr_wk',
        'weekday',
        'wday',
        'month',
        'year'
    ]
]

df['d'] = 'd_' + df['day'].astype(str)

df = df.merge(
    calendar_small,
    on='d',
    how='left'
)

df['date'] = pd.to_datetime(df['date'])

print(df.shape)

df.head()

(200865, 19)


,id,item_id,dept_id,cat_id,store_id,state_id,avg_demand,std_demand,cv,zero_demand_ratio,sales,day,d,date,wm_yr_wk,weekday,wday,month,year
0,FOODS_1_022_CA_4_validation,FOODS_1_022,FOODS_1,FOODS,CA_4,CA,0.095661,0.398411,4.164373,0.928385,0,1,d_1,2011-01-29,11101,Saturday,1,1,2011
1,FOODS_1_022_CA_4_validation,FOODS_1_022,FOODS_1,FOODS,CA_4,CA,0.095661,0.398411,4.164373,0.928385,0,2,d_2,2011-01-30,11101,Sunday,2,1,2011
2,FOODS_1_022_CA_4_validation,FOODS_1_022,FOODS_1,FOODS,CA_4,CA,0.095661,0.398411,4.164373,0.928385,0,3,d_3,2011-01-31,11101,Monday,3,1,2011
3,FOODS_1_022_CA_4_validation,FOODS_1_022,FOODS_1,FOODS,CA_4,CA,0.095661,0.398411,4.164373,0.928385,0,4,d_4,2011-02-01,11101,Tuesday,4,2,2011
4,FOODS_1_022_CA_4_validation,FOODS_1_022,FOODS_1,FOODS,CA_4,CA,0.095661,0.398411,4.164373,0.928385,0,5,d_5,2011-02-02,11101,Wednesday,5,2,2011


The long-format sales data was enriched using the M5 calendar dataset. This allows the forecasting model to capture temporal effects such as weekdays, weekends, monthly seasonality, and yearly trends. Calendar-based variables are often strong predictors of retail demand.

In [ ]:
print("Creating temporal features...")

df['day_of_week'] = df['date'].dt.dayofweek

df['is_weekend'] = (
    df['day_of_week']
    .isin([5, 6])
    .astype(int)
)

df['month_num'] = df['date'].dt.month

df['quarter'] = df['date'].dt.quarter

df['year_num'] = df['date'].dt.year

df[
    [
        'date',
        'day_of_week',
        'is_weekend',
        'month_num',
        'quarter',
        'year_num'
    ]
].head()

Creating temporal features...


,date,day_of_week,is_weekend,month_num,quarter,year_num
0,2011-01-29,5,1,1,1,2011
1,2011-01-30,6,1,1,1,2011
2,2011-01-31,0,0,1,1,2011
3,2011-02-01,1,0,2,1,2011
4,2011-02-02,2,0,2,1,2011


Temporal features were generated from the calendar information. These features help capture periodic retail demand patterns such as weekly purchasing cycles, weekend effects, and seasonal trends that are difficult to learn directly from raw sales values.

In [ ]:
print("Creating lag features...")

df = df.sort_values(
    ['id', 'day']
)

df['lag_1'] = (
    df.groupby('id')['sales']
    .shift(1)
)

df['lag_7'] = (
    df.groupby('id')['sales']
    .shift(7)
)

df['lag_28'] = (
    df.groupby('id')['sales']
    .shift(28)
)

df[
    [
        'sales',
        'lag_1',
        'lag_7',
        'lag_28'
    ]
].head(35)

Creating lag features...


,sales,lag_1,lag_7,lag_28
0,0,NaN,NaN,NaN
1,0,0.0,NaN,NaN
2,0,0.0,NaN,NaN
3,0,0.0,NaN,NaN
4,0,0.0,NaN,NaN
5,0,0.0,NaN,NaN
6,0,0.0,NaN,NaN
7,0,0.0,0.0,NaN
8,0,0.0,0.0,NaN
9,0,0.0,0.0,NaN


Lag features capture historical demand information. For example, lag_1 represents the previous day's sales, while lag_7 and lag_28 capture weekly and monthly purchasing behavior. These variables provide the model with memory of past demand patterns.

Lag features were generated to provide historical demand information to forecasting models. Missing values at the beginning of each SKU series are expected because sufficient historical observations are not yet available. These rows will be removed before model training.

In [ ]:
# Creating rolling features
# Rolling mean/std helps the model understand recent demand behaviour

print("Creating rolling features...")

df['rolling_mean_7'] = df.groupby('id')['sales'].transform(
    lambda x: x.shift(1).rolling(7).mean()
)

df['rolling_mean_28'] = df.groupby('id')['sales'].transform(
    lambda x: x.shift(1).rolling(28).mean()
)

df['rolling_std_7'] = df.groupby('id')['sales'].transform(
    lambda x: x.shift(1).rolling(7).std()
)

df['rolling_std_28'] = df.groupby('id')['sales'].transform(
    lambda x: x.shift(1).rolling(28).std()
)

df[['id', 'day', 'sales', 'rolling_mean_7', 'rolling_mean_28',
    'rolling_std_7', 'rolling_std_28']].head(35)

Creating rolling features...


,id,day,sales,rolling_mean_7,rolling_mean_28,rolling_std_7,rolling_std_28
0,FOODS_1_022_CA_4_validation,1,0,NaN,NaN,NaN,NaN
1,FOODS_1_022_CA_4_validation,2,0,NaN,NaN,NaN,NaN
2,FOODS_1_022_CA_4_validation,3,0,NaN,NaN,NaN,NaN
3,FOODS_1_022_CA_4_validation,4,0,NaN,NaN,NaN,NaN
4,FOODS_1_022_CA_4_validation,5,0,NaN,NaN,NaN,NaN
5,FOODS_1_022_CA_4_validation,6,0,NaN,NaN,NaN,NaN
6,FOODS_1_022_CA_4_validation,7,0,NaN,NaN,NaN,NaN
7,FOODS_1_022_CA_4_validation,8,0,0.0,NaN,0.0,NaN
8,FOODS_1_022_CA_4_validation,9,0,0.0,NaN,0.0,NaN
9,FOODS_1_022_CA_4_validation,10,0,0.0,NaN,0.0,NaN


In [ ]:
# Removing rows where lag/rolling values are missing

df_features = df.dropna()

print("Original shape:", df.shape)
print("After removing missing values:", df_features.shape)

df_features.head()

Original shape: (200865, 31)
After removing missing values: (197925, 31)


,id,item_id,dept_id,cat_id,store_id,state_id,avg_demand,std_demand,cv,zero_demand_ratio,...,month_num,quarter,year_num,lag_1,lag_7,lag_28,rolling_mean_7,rolling_mean_28,rolling_std_7,rolling_std_28
28,FOODS_1_022_CA_4_validation,FOODS_1_022,FOODS_1,FOODS,CA_4,CA,0.095661,0.398411,4.164373,0.928385,...,2,1,2011,0.0,0.0,0.0,0.0,0.0,0.0,0.0
29,FOODS_1_022_CA_4_validation,FOODS_1_022,FOODS_1,FOODS,CA_4,CA,0.095661,0.398411,4.164373,0.928385,...,2,1,2011,0.0,0.0,0.0,0.0,0.0,0.0,0.0
30,FOODS_1_022_CA_4_validation,FOODS_1_022,FOODS_1,FOODS,CA_4,CA,0.095661,0.398411,4.164373,0.928385,...,2,1,2011,0.0,0.0,0.0,0.0,0.0,0.0,0.0
31,FOODS_1_022_CA_4_validation,FOODS_1_022,FOODS_1,FOODS,CA_4,CA,0.095661,0.398411,4.164373,0.928385,...,3,1,2011,0.0,0.0,0.0,0.0,0.0,0.0,0.0
32,FOODS_1_022_CA_4_validation,FOODS_1_022,FOODS_1,FOODS,CA_4,CA,0.095661,0.398411,4.164373,0.928385,...,3,1,2011,0.0,0.0,0.0,0.0,0.0,0.0,0.0


In [ ]:
# Saving final feature-engineered dataset

df_features.to_csv("data/processed/features_phase2.csv", index=False)

print("Feature engineered data saved successfully!")

Feature engineered data saved successfully!


In [ ]:
# Check missing values in important feature columns

feature_cols = [
    'lag_1', 'lag_7', 'lag_28',
    'rolling_mean_7', 'rolling_mean_28',
    'rolling_std_7', 'rolling_std_28',
    'day_of_week', 'is_weekend', 'month_num'
]

df[feature_cols].isnull().sum()

,0
lag_1,105
lag_7,735
lag_28,2940
rolling_mean_7,735
rolling_mean_28,2940
rolling_std_7,735
rolling_std_28,2940
day_of_week,0
is_weekend,0
month_num,0


In [ ]:
# Remove rows with missing feature values

df_features = df.dropna().reset_index(drop=True)

print("Before:", df.shape)
print("After:", df_features.shape)

df_features.head()

Before: (200865, 31)
After: (197925, 31)


,id,item_id,dept_id,cat_id,store_id,state_id,avg_demand,std_demand,cv,zero_demand_ratio,...,month_num,quarter,year_num,lag_1,lag_7,lag_28,rolling_mean_7,rolling_mean_28,rolling_std_7,rolling_std_28
0,FOODS_1_022_CA_4_validation,FOODS_1_022,FOODS_1,FOODS,CA_4,CA,0.095661,0.398411,4.164373,0.928385,...,2,1,2011,0.0,0.0,0.0,0.0,0.0,0.0,0.0
1,FOODS_1_022_CA_4_validation,FOODS_1_022,FOODS_1,FOODS,CA_4,CA,0.095661,0.398411,4.164373,0.928385,...,2,1,2011,0.0,0.0,0.0,0.0,0.0,0.0,0.0
2,FOODS_1_022_CA_4_validation,FOODS_1_022,FOODS_1,FOODS,CA_4,CA,0.095661,0.398411,4.164373,0.928385,...,2,1,2011,0.0,0.0,0.0,0.0,0.0,0.0,0.0
3,FOODS_1_022_CA_4_validation,FOODS_1_022,FOODS_1,FOODS,CA_4,CA,0.095661,0.398411,4.164373,0.928385,...,3,1,2011,0.0,0.0,0.0,0.0,0.0,0.0,0.0
4,FOODS_1_022_CA_4_validation,FOODS_1_022,FOODS_1,FOODS,CA_4,CA,0.095661,0.398411,4.164373,0.928385,...,3,1,2011,0.0,0.0,0.0,0.0,0.0,0.0,0.0


In [ ]:
df_features.to_csv("data/processed/features_phase2.csv", index=False)

print("features_phase2.csv saved successfully!")

features_phase2.csv saved successfully!


In [ ]:
!ls data/processed

features_phase2.csv  sample_100_skus.csv  sample_100_skus_long.csv


In [ ]:
df_features = df.dropna().reset_index(drop=True)

print("Before:", df.shape)
print("After:", df_features.shape)

df_features.head()

Before: (200865, 31)
After: (197925, 31)


,id,item_id,dept_id,cat_id,store_id,state_id,avg_demand,std_demand,cv,zero_demand_ratio,...,month_num,quarter,year_num,lag_1,lag_7,lag_28,rolling_mean_7,rolling_mean_28,rolling_std_7,rolling_std_28
0,FOODS_1_022_CA_4_validation,FOODS_1_022,FOODS_1,FOODS,CA_4,CA,0.095661,0.398411,4.164373,0.928385,...,2,1,2011,0.0,0.0,0.0,0.0,0.0,0.0,0.0
1,FOODS_1_022_CA_4_validation,FOODS_1_022,FOODS_1,FOODS,CA_4,CA,0.095661,0.398411,4.164373,0.928385,...,2,1,2011,0.0,0.0,0.0,0.0,0.0,0.0,0.0
2,FOODS_1_022_CA_4_validation,FOODS_1_022,FOODS_1,FOODS,CA_4,CA,0.095661,0.398411,4.164373,0.928385,...,2,1,2011,0.0,0.0,0.0,0.0,0.0,0.0,0.0
3,FOODS_1_022_CA_4_validation,FOODS_1_022,FOODS_1,FOODS,CA_4,CA,0.095661,0.398411,4.164373,0.928385,...,3,1,2011,0.0,0.0,0.0,0.0,0.0,0.0,0.0
4,FOODS_1_022_CA_4_validation,FOODS_1_022,FOODS_1,FOODS,CA_4,CA,0.095661,0.398411,4.164373,0.928385,...,3,1,2011,0.0,0.0,0.0,0.0,0.0,0.0,0.0


In [ ]:
df_features.to_csv(
    "data/processed/features_phase2.csv",
    index=False
)

print("Feature engineering completed successfully!")

Feature engineering completed successfully!


In [ ]:
!ls data/processed

features_phase2.csv  sample_100_skus.csv  sample_100_skus_long.csv


eature Engineering Summary
Calendar information was merged with the sales data.
Temporal features such as day of week, weekend indicator, month, quarter, and year were generated.
Lag features (1, 7, and 28 days) were created to capture historical demand patterns.
Rolling mean and rolling standard deviation features were generated to capture recent demand trends and volatility.
Rows with insufficient historical information were removed.
The final dataset is now suitable for machine learning forecasting models.

In [ ]:
!git status

On branch main
Your branch is ahead of 'origin/main' by 1 commit.
  (use "git push" to publish your local commits)

Changes not staged for commit:
  (use "git add <file>..." to update what will be committed)
  (use "git restore <file>..." to discard changes in working directory)
	modified:   notebooks/00_project_setup.ipynb

no changes added to commit (use "git add" and/or "git commit -a")


In [ ]:
!find notebooks -name "*.ipynb"

notebooks/01_demand_analysis.ipynb
notebooks/00_project_setup.ipynb


In [ ]:
!ls -lh data/processed

total 81M
-rw------- 1 root root  51M May 29 19:33 features_phase2.csv
-rw------- 1 root root 414K May 23 11:11 sample_100_skus.csv
-rw------- 1 root root  29M May 26 14:41 sample_100_skus_long.csv


In [ ]:
!find notebooks -name "*.ipynb"

find: ‘notebooks’: No such file or directory


In [1]:
from google.colab import drive
drive.mount('/content/drive')

%cd /content/drive/MyDrive/inventory-optimization-ml-or

Mounted at /content/drive
/content/drive/MyDrive/inventory-optimization-ml-or


In [2]:
!git fetch origin
!git reset --hard origin/main
!git status

HEAD is now at b8882f7 Add Phase 1 demand analysis notebook
On branch main
Your branch is up to date with 'origin/main'.

nothing to commit, working tree clean
